# 🧠 第一课：认识 Ontology — 从混乱到秩序

> **在这节课中，你将通过一个真实场景，理解 Ontology（本体论）是什么、它能解决什么问题、以及为什么 Palantir、Airbus、NHS 等大型组织都在用它。**

### 📋 学习目标

| # | 目标 | 状态 |
|---|------|------|
| 1 | 理解 Ontology 在信息科学中的核心定义 | ☐ |
| 2 | 认识 Ontology 与词汇表、分类法、Schema 的区别 | ☐ |
| 3 | 掌握五大构件：类、实例、属性、关系、公理 | ☐ |
| 4 | 理解 Ontology 对企业数据治理和 AI 的价值 | ☐ |

### 🗺️ 课程路线图

```
第一课(本课)          第二课              第三课              第四课              第五课
认识 Ontology  →  OPERA 建模框架  →  代码建模实战  →  多领域案例  →  Ontology × LLM
  [概念理解]         [方法论]           [动手编码]        [举一反三]       [前沿融合]
```

---
## 🎬 Part 1：场景引入 — 一家公司的"数据灾难"

> 想象你是一家快速成长的电商公司 **ShopFast** 的 CTO。公司有三个核心部门，各自维护着自己的数据库。今天你收到了一封投诉邮件...

**📧 CEO 邮件：**
> "为什么客服说订单 #10086 已经退款了，但财务说根本没有这笔退款记录？仓库那边说这个订单的货物还没发出去？我们的系统到底怎么了？！"

让我们看看三个部门的「订单」到底长什么样 👇

In [2]:
# 🏢 三个部门各自定义的"订单"——混乱的开始

# === 客服系统 ===
order_customer_service = {
    "工单号": "CS-10086",
    "客户姓名": "张三",
    "客户手机": "138xxxx1234",
    "问题类型": "退款",
    "状态": "已处理",           # ← 客服认为"已处理"
    "处理时间": "2025-01-15",
    "备注": "客户要求退货退款"
}

# === 仓储系统 ===
order_warehouse = {
    "order_id": "WH-20250115-003",   # ← 完全不同的编号！
    "sku_list": ["SKU-A001", "SKU-B023"],
    "warehouse": "华东仓",
    "ship_status": "pending",         # ← 仓库说还没发货
    "weight_kg": 2.5,
    "created": "2025-01-14"
}

# === 财务系统 ===
order_finance = {
    "invoice_no": "INV-2025-0042",    # ← 又一个不同的编号！
    "amount": 299.00,
    "currency": "CNY",
    "payment_status": "paid",          # ← 财务说已付款，没有退款
    "tax": 29.90,
    "channel": "支付宝"
}

print("=" * 60)
print("🏢 客服系统的订单：")
for k, v in order_customer_service.items():
    print(f"   {k}: {v}")

print("\n🏭 仓储系统的订单：")
for k, v in order_warehouse.items():
    print(f"   {k}: {v}")

print("\n💰 财务系统的订单：")
for k, v in order_finance.items():
    print(f"   {k}: {v}")

print("\n" + "=" * 60)
print("❌ 问题暴露：")
print("   1. 编号不统一（CS-xx / WH-xx / INV-xx）")
print("   2. 字段名不一致（状态 / ship_status / payment_status）")
print("   3. 语义冲突（'已处理' ≠ '已发货' ≠ '已退款'）")
print("   4. 无法跨系统关联查询——同一笔订单在三个系统中无法对应！")

🏢 客服系统的订单：
   工单号: CS-10086
   客户姓名: 张三
   客户手机: 138xxxx1234
   问题类型: 退款
   状态: 已处理
   处理时间: 2025-01-15
   备注: 客户要求退货退款

🏭 仓储系统的订单：
   order_id: WH-20250115-003
   sku_list: ['SKU-A001', 'SKU-B023']
   warehouse: 华东仓
   ship_status: pending
   weight_kg: 2.5
   created: 2025-01-14

💰 财务系统的订单：
   invoice_no: INV-2025-0042
   amount: 299.0
   currency: CNY
   payment_status: paid
   tax: 29.9
   channel: 支付宝

❌ 问题暴露：
   1. 编号不统一（CS-xx / WH-xx / INV-xx）
   2. 字段名不一致（状态 / ship_status / payment_status）
   3. 语义冲突（'已处理' ≠ '已发货' ≠ '已退款'）
   4. 无法跨系统关联查询——同一笔订单在三个系统中无法对应！


### ❓ 思考：如果你是 CTO，你会怎么解决这个问题？

问题的根源不是技术——不是数据库不好、不是接口不通。根源是：

> **三个部门对"订单"这个概念没有共识。**

我们需要的是：
1. ✅ **统一的概念定义** — 什么是"订单"？它有哪些属性？
2. ✅ **标准化的数据模型** — 所有系统用同一套字段名和数据结构
3. ✅ **可被机器理解的语义** — 不只是人能看懂，程序也能理解含义
4. ✅ **约束和规则** — "一个订单必须关联一个客户"、"金额必须大于0"

**这，正是 Ontology（本体论）要解决的问题！** 🎯

---
## 📖 Part 2：什么是 Ontology？

### 从哲学到信息科学

| 时代 | 含义 | 代表 |
|------|------|------|
| 古希腊 | 研究"存在的本质"的哲学分支 | 亚里士多德 |
| 1990s | 信息科学中对领域知识的形式化建模方法 | Tom Gruber |
| 2010s+ | 企业数据的统一语义层，AI 的知识骨架 | Palantir, Google KG |

### 📌 信息科学中的经典定义

> **Tom Gruber (1993):**
> 
> *"An ontology is a formal, explicit specification of a shared conceptualization."*
> 
> **翻译：** Ontology 是对**共享概念体系**的**形式化**、**明确的**规范说明。

让我们拆解这个定义的四个关键词 👇

In [3]:
# 🔍 交互式理解 Ontology 的四个关键词

definition_keywords = {
    "Formal（形式化）": {
        "含义": "可以被计算机处理和推理，不是自然语言的模糊描述",
        "例子": "用 OWL 语言定义'Dog is-a Animal'，推理引擎能自动推导"
    },
    "Explicit（明确的）": {
        "含义": "每个概念和约束都有清晰的定义，没有歧义",
        "例子": "'订单金额'明确是浮点数，单位是人民币，不能为负数"
    },
    "Shared（共享的）": {
        "含义": "是团队/组织达成共识的模型，不是个人视角",
        "例子": "客服、仓储、财务都认同'订单'的定义和字段"
    },
    "Conceptualization（概念化）": {
        "含义": "对真实世界某个领域的抽象模型",
        "例子": "把电商业务抽象为：客户、订单、商品、支付 等概念"
    }
}

print("📖 Ontology 定义的四个关键词\n")
for i, (keyword, info) in enumerate(definition_keywords.items(), 1):
    print(f"  {i}. {keyword}")
    print(f"     含义：{info['含义']}")
    print(f"     🔹 例子：{info['例子']}")
    print()

print("=" * 60)
print("🎯 一句话总结：")
print("   Ontology = 大家都认同的 + 机器能理解的 + 关于某个领域的 + 概念模型")

📖 Ontology 定义的四个关键词

  1. Formal（形式化）
     含义：可以被计算机处理和推理，不是自然语言的模糊描述
     🔹 例子：用 OWL 语言定义'Dog is-a Animal'，推理引擎能自动推导

  2. Explicit（明确的）
     含义：每个概念和约束都有清晰的定义，没有歧义
     🔹 例子：'订单金额'明确是浮点数，单位是人民币，不能为负数

  3. Shared（共享的）
     含义：是团队/组织达成共识的模型，不是个人视角
     🔹 例子：客服、仓储、财务都认同'订单'的定义和字段

  4. Conceptualization（概念化）
     含义：对真实世界某个领域的抽象模型
     🔹 例子：把电商业务抽象为：客户、订单、商品、支付 等概念

🎯 一句话总结：
   Ontology = 大家都认同的 + 机器能理解的 + 关于某个领域的 + 概念模型


---
## 🔍 Part 3：Ontology 和其他概念有什么区别？

你可能听说过词汇表、分类法、Schema、知识图谱——它们和 Ontology 是什么关系？

> **它们不是互相替代的，而是层层递进的语义能力阶梯：**

In [4]:
# 📊 语义能力阶梯——从低到高

levels = [
    {
        "name": "词汇表 (Vocabulary)",
        "能力": "统一术语名称",
        "例子": "'customer' = '客户'",
        "限制": "只有名字，没有结构"
    },
    {
        "name": "分类法 (Taxonomy)",
        "能力": "层级分类关系 (is-a)",
        "例子": "哺乳动物 → 犬科 → 狗",
        "限制": "只有上下级，没有其他关系"
    },
    {
        "name": "Schema (数据模式)",
        "能力": "定义字段类型和约束",
        "例子": "Order { id: int, amount: float }",
        "限制": "偏技术实现，缺乏语义"
    },
    {
        "name": "知识图谱 (Knowledge Graph)",
        "能力": "实体 + 关系的大规模网络",
        "例子": "Google KG：北京 → 首都 → 中国",
        "限制": "偏数据层，缺乏严格推理规则"
    },
    {
        "name": "✨ Ontology (本体)",
        "能力": "概念 + 属性 + 关系 + 公理 + 推理",
        "例子": "完整的领域模型，可自动推理",
        "限制": "构建成本较高（但非常值得）"
    }
]

print("📊 语义能力阶梯\n")
bar_width = 30
for i, level in enumerate(levels):
    filled = int(bar_width * (i + 1) / len(levels))
    bar = "█" * filled + "░" * (bar_width - filled)
    marker = " 👈 我们的目标！" if i == len(levels) - 1 else ""
    print(f"  Level {i+1}: {level['name']}{marker}")
    print(f"          [{bar}]")
    print(f"          能力：{level['能力']}")
    print(f"          例子：{level['例子']}")
    print(f"          局限：{level['限制']}")
    print()

print("💡 关键认知：Ontology 是语义能力的最高层级。")
print("   知识图谱是'数据'，Ontology 是知识图谱的'定义和规则'。")

📊 语义能力阶梯

  Level 1: 词汇表 (Vocabulary)
          [██████░░░░░░░░░░░░░░░░░░░░░░░░]
          能力：统一术语名称
          例子：'customer' = '客户'
          局限：只有名字，没有结构

  Level 2: 分类法 (Taxonomy)
          [████████████░░░░░░░░░░░░░░░░░░]
          能力：层级分类关系 (is-a)
          例子：哺乳动物 → 犬科 → 狗
          局限：只有上下级，没有其他关系

  Level 3: Schema (数据模式)
          [██████████████████░░░░░░░░░░░░]
          能力：定义字段类型和约束
          例子：Order { id: int, amount: float }
          局限：偏技术实现，缺乏语义

  Level 4: 知识图谱 (Knowledge Graph)
          [████████████████████████░░░░░░]
          能力：实体 + 关系的大规模网络
          例子：Google KG：北京 → 首都 → 中国
          局限：偏数据层，缺乏严格推理规则

  Level 5: ✨ Ontology (本体) 👈 我们的目标！
          [██████████████████████████████]
          能力：概念 + 属性 + 关系 + 公理 + 推理
          例子：完整的领域模型，可自动推理
          局限：构建成本较高（但非常值得）

💡 关键认知：Ontology 是语义能力的最高层级。
   知识图谱是'数据'，Ontology 是知识图谱的'定义和规则'。


---
## 🧱 Part 4：Ontology 的五大构件

Ontology 的核心就五样东西。掌握了这五个概念，你就掌握了 Ontology 的 80%。

| # | 构件 | 英文 | 一句话解释 | 类比 |
|---|------|------|-----------|------|
| 1 | **类** | Class | 概念的分类模板 | Excel 的"表名" |
| 2 | **实例** | Individual | 具体的对象 | Excel 的"某一行数据" |
| 3 | **属性** | Property | 描述特征的字段 | Excel 的"列名" |
| 4 | **关系** | Relation | 概念之间的语义连接 | 两张表之间的外键 |
| 5 | **公理** | Axiom | 必须遵守的规则 | 数据库的约束条件 |

让我们用代码来构建一个 Ontology，直观感受这五个构件 👇

In [5]:
# 🧱 用 Python 从零构建一个简单的 Ontology 引擎

class SimpleOntology:
    """一个教学用的简化 Ontology 引擎，演示五大构件"""
    
    def __init__(self, name):
        self.name = name
        self.classes = {}          # 类 {name: {parent, description}}
        self.individuals = {}      # 实例 {name: {class, properties}}
        self.properties = {}       # 属性定义 {name: {domain, range, type}}
        self.relations = {}        # 关系定义 {name: {domain, range, desc}}
        self.axioms = []           # 公理/规则列表
        print(f"✅ 创建本体：{name}")
    
    # --- 构件 1：类 (Class) ---
    def add_class(self, name, parent=None, description=""):
        self.classes[name] = {"parent": parent, "description": description}
        parent_info = f" (继承自 {parent})" if parent else " (顶层类)"
        print(f"   📦 添加类：{name}{parent_info}")
    
    # --- 构件 2：实例 (Individual) ---
    def add_individual(self, name, cls, **properties):
        self.individuals[name] = {"class": cls, "properties": properties}
        print(f"   🏷️  添加实例：{name} (属于类 {cls})")
    
    # --- 构件 3：属性 (Property) ---
    def add_property(self, name, domain, range_type, prop_type="data"):
        self.properties[name] = {"domain": domain, "range": range_type, "type": prop_type}
        print(f"   📋 添加属性：{domain}.{name} → {range_type}")
    
    # --- 构件 4：关系 (Relation) ---
    def add_relation(self, name, domain, range_cls, description=""):
        self.relations[name] = {"domain": domain, "range": range_cls, "description": description}
        print(f"   🔗 添加关系：{domain} --[{name}]--> {range_cls}")
    
    # --- 构件 5：公理 (Axiom) ---
    def add_axiom(self, rule, description=""):
        self.axioms.append({"rule": rule, "description": description})
        print(f"   ⚖️  添加公理：{description}")
    
    # --- 查看本体结构 ---
    def summary(self):
        print(f"\n{'='*60}")
        print(f"📊 本体 [{self.name}] 概况")
        print(f"{'='*60}")
        print(f"   类 (Class):       {len(self.classes)} 个")
        print(f"   实例 (Individual): {len(self.individuals)} 个")
        print(f"   属性 (Property):  {len(self.properties)} 个")
        print(f"   关系 (Relation):  {len(self.relations)} 个")
        print(f"   公理 (Axiom):     {len(self.axioms)} 条")
        print(f"{'='*60}")

    # --- 推理引擎 ---
    def reason(self):
        """基于公理进行自动推理"""
        print("\n🤖 启动推理引擎...")
        inferred = []
        for axiom in self.axioms:
            rule = axiom["rule"]
            for name, ind in self.individuals.items():
                result = rule(name, ind)
                if result:
                    inferred.append(result)
                    print(f"   💡 推理发现：{result}")
        if not inferred:
            print("   (未发现新的推理结果)")
        return inferred

print("✅ SimpleOntology 类定义完成！下面我们来使用它。")

✅ SimpleOntology 类定义完成！下面我们来使用它。


### 🐾 动手实践：构建「宠物医院」本体

让我们用刚定义的 `SimpleOntology` 类，为一家小型宠物医院建模。

我们要描述的世界包括：
- 🐶🐱 不同种类的动物
- 👨‍⚕️ 兽医
- 💊 治疗和诊断
- 以及它们之间的关系和规则

In [10]:
# 🐾 构建「宠物医院」本体

onto = SimpleOntology("宠物医院")

# === 构件 1：定义类 (Class) ===
print("\n--- 第一步：定义类层次 ---")
onto.add_class("Animal", description="所有动物的基类")
onto.add_class("Dog", parent="Animal", description="犬类")
onto.add_class("Cat", parent="Animal", description="猫类")
onto.add_class("Person", description="人")
onto.add_class("Vet", parent="Person", description="兽医")
onto.add_class("Diagnosis", description="诊断结果")
onto.add_class("Treatment", description="治疗方案")

# === 构件 2：定义属性 (Property) ===
print("\n--- 第二步：定义属性 ---")
onto.add_property("name", "Animal", "string")
onto.add_property("age", "Animal", "integer")
onto.add_property("weight_kg", "Animal", "float")
onto.add_property("vaccinated", "Animal", "boolean")
onto.add_property("specialty", "Vet", "string")

# === 构件 3：定义关系 (Relation) ===
print("\n--- 第三步：定义关系 ---")
onto.add_relation("treated_by", "Animal", "Vet", "动物由某兽医治疗")
onto.add_relation("has_diagnosis", "Animal", "Diagnosis", "动物有某个诊断")
onto.add_relation("requires_treatment", "Diagnosis", "Treatment", "诊断需要某种治疗")

# === 构件 4：添加实例 (Individual) ===
print("\n--- 第四步：添加实例 ---")
onto.add_individual("旺财", "Dog", age=3, weight_kg=12.5, vaccinated=True)
onto.add_individual("咪咪", "Cat", age=1, weight_kg=4.2, vaccinated=False)
onto.add_individual("大黄", "Dog", age=8, weight_kg=25.0, vaccinated=True)
onto.add_individual("李医生", "Vet", specialty="骨科")
onto.add_individual("王医生", "Vet", specialty="内科")
onto.add_individual("骨折", "Diagnosis")
onto.add_individual("手术", "Treatment")

# === 构件 5：定义公理 (Axiom) ===
print("\n--- 第五步：定义公理（规则）---")

# 公理1：Dog 且年龄 < 1 → 幼犬
onto.add_axiom(
    rule=lambda name, ind: f"{name} 是幼犬（< 1岁）" 
        if ind["class"] == "Dog" and ind["properties"].get("age", 99) < 1 else None,
    description="如果是 Dog 且年龄 < 1 → 标记为幼犬"
)

# 公理2：未接种疫苗 → 需要疫苗提醒
onto.add_axiom(
    rule=lambda name, ind: f"⚠️ {name} 未接种疫苗，需要安排接种！"
        if ind["class"] in ("Dog", "Cat") and not ind["properties"].get("vaccinated", True) else None,
    description="如果动物未接种疫苗 → 发出接种提醒"
)

# 公理3：大型犬（>20kg）→ 需要额外关注
onto.add_axiom(
    rule=lambda name, ind: f"🐕 {name} 是大型犬（{ind['properties'].get('weight_kg', 0)}kg），需要大型犬专用候诊区"
        if ind["class"] == "Dog" and ind["properties"].get("weight_kg", 0) > 20 else None,
    description="如果是 Dog 且体重 > 20kg → 大型犬标记"
)

# 查看概况
onto.summary()

✅ 创建本体：宠物医院

--- 第一步：定义类层次 ---
   📦 添加类：Animal (顶层类)
   📦 添加类：Dog (继承自 Animal)
   📦 添加类：Cat (继承自 Animal)
   📦 添加类：Person (顶层类)
   📦 添加类：Vet (继承自 Person)
   📦 添加类：Diagnosis (顶层类)
   📦 添加类：Treatment (顶层类)

--- 第二步：定义属性 ---
   📋 添加属性：Animal.name → string
   📋 添加属性：Animal.age → integer
   📋 添加属性：Animal.weight_kg → float
   📋 添加属性：Animal.vaccinated → boolean
   📋 添加属性：Vet.specialty → string

--- 第三步：定义关系 ---
   🔗 添加关系：Animal --[treated_by]--> Vet
   🔗 添加关系：Animal --[has_diagnosis]--> Diagnosis
   🔗 添加关系：Diagnosis --[requires_treatment]--> Treatment

--- 第四步：添加实例 ---
   🏷️  添加实例：旺财 (属于类 Dog)
   🏷️  添加实例：咪咪 (属于类 Cat)
   🏷️  添加实例：大黄 (属于类 Dog)
   🏷️  添加实例：李医生 (属于类 Vet)
   🏷️  添加实例：王医生 (属于类 Vet)
   🏷️  添加实例：骨折 (属于类 Diagnosis)
   🏷️  添加实例：手术 (属于类 Treatment)

--- 第五步：定义公理（规则）---
   ⚖️  添加公理：如果是 Dog 且年龄 < 1 → 标记为幼犬
   ⚖️  添加公理：如果动物未接种疫苗 → 发出接种提醒
   ⚖️  添加公理：如果是 Dog 且体重 > 20kg → 大型犬标记

📊 本体 [宠物医院] 概况
   类 (Class):       7 个
   实例 (Individual): 7 个
   属性 (Property):  5 个
   关系 (Relation):  3 个
   公理 

In [11]:
# 🤖 运行推理引擎——体验公理的自动推理能力

print("现在让我们看看，公理能自动发现什么信息：\n")
results = onto.reason()

print(f"\n✅ 推理完成！共发现 {len(results)} 条自动推理结果。")
print("\n💡 关键认知：这些结果不是我们手动标注的，")
print("   而是推理引擎根据 [数据 + 公理] 自动推导出来的！")
print("   这就是 Ontology '自动推理'能力的威力。")

现在让我们看看，公理能自动发现什么信息：


🤖 启动推理引擎...
   💡 推理发现：⚠️ 咪咪 未接种疫苗，需要安排接种！
   💡 推理发现：🐕 大黄 是大型犬（25.0kg），需要大型犬专用候诊区

✅ 推理完成！共发现 2 条自动推理结果。

💡 关键认知：这些结果不是我们手动标注的，
   而是推理引擎根据 [数据 + 公理] 自动推导出来的！
   这就是 Ontology '自动推理'能力的威力。


---
## 💡 Part 5：回到开篇 — 用 Ontology 解决电商数据灾难

还记得 ShopFast 的三个系统吗？让我们用 Ontology 的思维来统一它们：

**建模思路：**
1. 定义统一的**类**：Order、Customer、Product、Payment
2. 统一**属性**命名：不再有"工单号"vs"order_id"vs"invoice_no"
3. 建立**关系**：Order → placed_by → Customer
4. 设置**公理**：每个 Order 必须关联一个 Customer，金额 > 0

In [12]:
# 💡 为 ShopFast 建立统一的电商本体

ecommerce = SimpleOntology("ShopFast 电商本体")

# 统一的类定义
print("\n--- 统一类定义 ---")
ecommerce.add_class("Order", description="订单——三个系统的核心统一概念")
ecommerce.add_class("Customer", description="客户")
ecommerce.add_class("Product", description="商品")
ecommerce.add_class("Payment", description="支付记录")
ecommerce.add_class("Shipment", description="物流记录")

# 统一的属性
print("\n--- 统一属性（三个系统都使用这套字段名）---")
ecommerce.add_property("order_id", "Order", "string")
ecommerce.add_property("total_amount", "Order", "float")
ecommerce.add_property("status", "Order", "enum[pending, paid, shipped, delivered, refunded]")
ecommerce.add_property("created_at", "Order", "datetime")
ecommerce.add_property("customer_name", "Customer", "string")
ecommerce.add_property("phone", "Customer", "string")
ecommerce.add_property("sku", "Product", "string")
ecommerce.add_property("payment_method", "Payment", "string")
ecommerce.add_property("tracking_no", "Shipment", "string")

# 统一的关系
print("\n--- 统一关系 ---")
ecommerce.add_relation("placed_by", "Order", "Customer", "订单由客户下达")
ecommerce.add_relation("contains", "Order", "Product", "订单包含商品")
ecommerce.add_relation("paid_via", "Order", "Payment", "订单通过某方式支付")
ecommerce.add_relation("shipped_as", "Order", "Shipment", "订单对应某物流单")

# 公理/规则
print("\n--- 业务规则 (公理) ---")
ecommerce.add_axiom(
    rule=lambda name, ind: f"❌ {name}: 金额异常 (≤0)"
        if ind["class"] == "Order" and ind["properties"].get("total_amount", 1) <= 0 else None,
    description="订单金额必须 > 0"
)
ecommerce.add_axiom(
    rule=lambda name, ind: f"⚠️ {name}: 已支付但未发货超过3天"
        if ind["class"] == "Order" and ind["properties"].get("status") == "paid" 
        and ind["properties"].get("days_since_paid", 0) > 3 else None,
    description="已付款超过3天未发货 → 预警"
)

ecommerce.summary()

✅ 创建本体：ShopFast 电商本体

--- 统一类定义 ---
   📦 添加类：Order (顶层类)
   📦 添加类：Customer (顶层类)
   📦 添加类：Product (顶层类)
   📦 添加类：Payment (顶层类)
   📦 添加类：Shipment (顶层类)

--- 统一属性（三个系统都使用这套字段名）---
   📋 添加属性：Order.order_id → string
   📋 添加属性：Order.total_amount → float
   📋 添加属性：Order.status → enum[pending, paid, shipped, delivered, refunded]
   📋 添加属性：Order.created_at → datetime
   📋 添加属性：Customer.customer_name → string
   📋 添加属性：Customer.phone → string
   📋 添加属性：Product.sku → string
   📋 添加属性：Payment.payment_method → string
   📋 添加属性：Shipment.tracking_no → string

--- 统一关系 ---
   🔗 添加关系：Order --[placed_by]--> Customer
   🔗 添加关系：Order --[contains]--> Product
   🔗 添加关系：Order --[paid_via]--> Payment
   🔗 添加关系：Order --[shipped_as]--> Shipment

--- 业务规则 (公理) ---
   ⚖️  添加公理：订单金额必须 > 0
   ⚖️  添加公理：已付款超过3天未发货 → 预警

📊 本体 [ShopFast 电商本体] 概况
   类 (Class):       5 个
   实例 (Individual): 0 个
   属性 (Property):  9 个
   关系 (Relation):  4 个
   公理 (Axiom):     2 条


In [13]:
# 🔄 三个系统到统一本体的映射

print("🔄 字段映射表：三个系统 → 统一本体\n")
print(f"{'统一字段':<15} {'客服系统':<15} {'仓储系统':<18} {'财务系统':<15}")
print("-" * 65)

mapping = [
    ("order_id",     "工单号",       "order_id",        "invoice_no"),
    ("status",       "状态",         "ship_status",     "payment_status"),
    ("created_at",   "处理时间",     "created",         "—"),
    ("total_amount", "—",            "—",               "amount"),
    ("customer",     "客户姓名",     "—",               "—"),
    ("products",     "—",            "sku_list",        "—"),
    ("payment",      "—",            "—",               "channel"),
    ("shipping",     "—",            "warehouse",       "—"),
]

for unified, cs, wh, fin in mapping:
    print(f"{unified:<15} {cs:<15} {wh:<18} {fin:<15}")

print("\n" + "=" * 65)
print("✅ 有了统一的 Ontology，三个系统的数据可以自动对齐！")
print("\n💡 现在 CEO 的问题可以直接回答了：")
print("   查询 Order(order_id='10086')")
print("   → status: 'paid' (已付款，未退款)")
print("   → ship_status: 'pending' (未发货)")
print("   → 客服的'已处理'只是关了工单，不等于已退款")
print("   → 结论：需要客服跟进退款流程并通知仓库取消发货")

🔄 字段映射表：三个系统 → 统一本体

统一字段            客服系统            仓储系统               财务系统           
-----------------------------------------------------------------
order_id        工单号             order_id           invoice_no     
status          状态              ship_status        payment_status 
created_at      处理时间            created            —              
total_amount    —               —                  amount         
customer        客户姓名            —                  —              
products        —               sku_list           —              
payment         —               —                  channel        
shipping        —               warehouse          —              

✅ 有了统一的 Ontology，三个系统的数据可以自动对齐！

💡 现在 CEO 的问题可以直接回答了：
   查询 Order(order_id='10086')
   → status: 'paid' (已付款，未退款)
   → ship_status: 'pending' (未发货)
   → 客服的'已处理'只是关了工单，不等于已退款
   → 结论：需要客服跟进退款流程并通知仓库取消发货


---
## 🎯 Part 6：Ontology 能带来什么价值？

### 六大核心价值

| 价值 | 说明 | 实例 |
|------|------|------|
| 🗣️ **统一语义** | 消灭"鸡同鸭讲"，所有人用同一套概念 | 三个系统对"订单状态"达成共识 |
| 🔗 **数据互操作** | 异构系统"说同一种语言" | 客服、仓储、财务数据可关联查询 |
| ♻️ **知识重用** | 建一次，用多处 | 电商本体可复用到其他电商公司 |
| 🤖 **自动推理** | 机器自动发现隐含知识 | "未接种疫苗的动物"自动被标记 |
| 🧠 **AI 增强** | 为 LLM/Agent 提供知识骨架 | Palantir AIP 用 Ontology 驱动 AI |
| 📜 **治理合规** | 规则自动执行，违规自动告警 | "订单金额不能为负" 自动校验 |

### 🏢 谁在用 Ontology？

- **Palantir** — Foundry/AIP 的核心是 Ontology 层，连接军事、金融、医疗数据
- **NHS (英国国家卫生服务)** — 用 Ontology 管理新冠疫苗分发调度
- **Airbus** — 用 Ontology 统一全球供应链数据
- **Google** — Knowledge Graph 背后有 Schema.org Ontology
- **各大药企** — 用生物医学 Ontology 管理药物、疾病、基因关系

In [14]:
# 🤖 自动推理演示：给电商本体加入数据并推理

print("--- 为电商本体添加测试数据 ---\n")

# 添加订单实例
ecommerce.add_individual("ORD-001", "Order", 
    order_id="ORD-001", total_amount=299.0, status="paid", days_since_paid=5)
ecommerce.add_individual("ORD-002", "Order",
    order_id="ORD-002", total_amount=0, status="pending")
ecommerce.add_individual("ORD-003", "Order",
    order_id="ORD-003", total_amount=1599.0, status="shipped")

# 运行推理
results = ecommerce.reason()

print(f"\n✅ 自动检测出 {len(results)} 个问题/预警！")
print("\n这些结果完全是公理自动推导的——不需要人工逐个检查！")

--- 为电商本体添加测试数据 ---

   🏷️  添加实例：ORD-001 (属于类 Order)
   🏷️  添加实例：ORD-002 (属于类 Order)
   🏷️  添加实例：ORD-003 (属于类 Order)

🤖 启动推理引擎...
   💡 推理发现：❌ ORD-002: 金额异常 (≤0)
   💡 推理发现：⚠️ ORD-001: 已支付但未发货超过3天

✅ 自动检测出 2 个问题/预警！

这些结果完全是公理自动推导的——不需要人工逐个检查！


---
## 📝 第一课小结

### ✅ 你已经学会了：

- [x] **Ontology 的定义**：对共享概念体系的形式化、明确的规范说明
- [x] **核心问题**：Ontology 解决的是"对同一概念没有共识"的问题
- [x] **五大构件**：类、实例、属性、关系、公理
- [x] **语义阶梯**：词汇表 → 分类法 → Schema → 知识图谱 → Ontology
- [x] **自动推理**：公理 + 数据 = 自动发现隐含知识
- [x] **六大价值**：统一语义、数据互操作、知识重用、自动推理、AI增强、治理合规

### 🔜 下一课预告

> **第二课：OPERA 万能建模框架** — 一个简单的五步框架，让你能对任何领域进行 Ontology 建模。
> 
> OPERA = **O**bject Types + **P**roperties + **E**vents/Actions + **R**elations + **A**xioms

In [1]:
# 📝 学习检测：3 道小测验

questions = [
    {
        "question": "Ontology 经典定义中，'Shared' 的含义是？",
        "options": {
            "A": "在多台服务器之间共享",
            "B": "开源的、免费的",
            "C": "团队/组织达成共识的，不是个人视角",
            "D": "数据可以被多个程序访问"
        },
        "answer": "C",
        "explanation": "'Shared' 指的是这个概念模型是社区或组织共同认可的，不是某个人的主观理解。"
    },
    {
        "question": "Ontology 和知识图谱 (Knowledge Graph) 的关系是？",
        "options": {
            "A": "完全相同的概念",
            "B": "Ontology 是知识图谱的定义和规则层，知识图谱是数据层",
            "C": "知识图谱包含 Ontology",
            "D": "两者没有关系"
        },
        "answer": "B",
        "explanation": "Ontology 定义了有什么类、什么属性、什么规则；知识图谱是在这些定义下存储的具体数据。"
    },
    {
        "question": "在宠物医院例子中，'旺财 是 Dog 的一个实例' 中的 'Dog' 是 Ontology 的哪个构件？",
        "options": {
            "A": "实例 (Individual)",
            "B": "属性 (Property)",
            "C": "类 (Class)",
            "D": "公理 (Axiom)"
        },
        "answer": "C",
        "explanation": "Dog 是一个类 (Class)，代表一种概念分类。旺财是 Dog 类的一个实例 (Individual)。"
    }
]

score = 0
print("=" * 50)
print("📝 第一课小测验（共 3 题）")
print("=" * 50)

for i, q in enumerate(questions, 1):
    print(f"\n{'─'*50}")
    print(f"第 {i} 题：{q['question']}\n")
    for key, val in q["options"].items():
        print(f"  {key}. {val}")
    
    user_answer = input(f"\n你的答案 (A/B/C/D)：").strip().upper()
    
    if user_answer == q["answer"]:
        print(f"✅ 正确！{q['explanation']}")
        score += 1
    else:
        print(f"❌ 不对哦，正确答案是 {q['answer']}。{q['explanation']}")

print(f"\n{'='*50}")
print(f"🏆 测验结果：{score}/{len(questions)}")
if score == 3:
    print("🎉 太棒了！你已经完全掌握了第一课的核心概念！")
elif score >= 2:
    print("👍 不错！建议回顾一下答错的部分。")
else:
    print("💪 建议重新阅读本课内容后再试一次。")
print(f"\n➡️ 准备好了吗？打开第二课继续学习 OPERA 建模框架吧！")

📝 第一课小测验（共 3 题）

──────────────────────────────────────────────────
第 1 题：Ontology 经典定义中，'Shared' 的含义是？

  A. 在多台服务器之间共享
  B. 开源的、免费的
  C. 团队/组织达成共识的，不是个人视角
  D. 数据可以被多个程序访问
❌ 不对哦，正确答案是 C。'Shared' 指的是这个概念模型是社区或组织共同认可的，不是某个人的主观理解。

──────────────────────────────────────────────────
第 2 题：Ontology 和知识图谱 (Knowledge Graph) 的关系是？

  A. 完全相同的概念
  B. Ontology 是知识图谱的定义和规则层，知识图谱是数据层
  C. 知识图谱包含 Ontology
  D. 两者没有关系
❌ 不对哦，正确答案是 B。Ontology 定义了有什么类、什么属性、什么规则；知识图谱是在这些定义下存储的具体数据。

──────────────────────────────────────────────────
第 3 题：在宠物医院例子中，'旺财 是 Dog 的一个实例' 中的 'Dog' 是 Ontology 的哪个构件？

  A. 实例 (Individual)
  B. 属性 (Property)
  C. 类 (Class)
  D. 公理 (Axiom)
❌ 不对哦，正确答案是 C。Dog 是一个类 (Class)，代表一种概念分类。旺财是 Dog 类的一个实例 (Individual)。

🏆 测验结果：0/3
💪 建议重新阅读本课内容后再试一次。

➡️ 准备好了吗？打开第二课继续学习 OPERA 建模框架吧！
